# 🎙️ Meeting Assistant: audio → transcript → minutes, decisions, action items

The whole app runs in this notebook: three model stages plus an interactive web interface.

| Stage | Model | Role |
|---|---|---|
| 1. Speech-to-text | **WhisperX** (Whisper `large-v3` + wav2vec2 alignment + pyannote diarization) | Raw transcript with speakers and timestamps |
| 2. Refinement (LLM 1) | **Gemini 3.1 Flash Lite** via LangChain | Fixes recognition errors in names, terms and acronyms using context, without adding content |
| 3. Meeting record (LLM 2) | **Qwen 2.5 72B Instruct** via OpenRouter | Minutes, key decisions and action items as Pydantic-validated JSON |

## How to run
1. **Runtime → Change runtime type → T4 GPU** (or any GPU) → Save.
2. Put your **OpenRouter API key** and other keys (GEMINI_API_KEY and HF_TOKEN) in the config cell (`OPENROUTER_API_KEY = "sk-or-..."`), or add it in Colab Secrets (🔑 on the left) as `OPENROUTER_API_KEY`. Stage 3 needs it.
3. **Runtime → Run all**.
4. The last cell shows the app and a public link like `https://xxxx.gradio.live`. Open it, upload a recording, press **Start processing**.

The first run installs packages and downloads the speech models, so it takes a few minutes. The first time, the install cell **restarts the session by itself** (Colab may say the session crashed; that is expected). Then click **Runtime → Run all** again and it runs straight through.

In [1]:
# ============================================================
# STEP 1 — INSTALL (takes a few minutes the first time)
# ============================================================
# pip replaces numpy/torch versions that Colab already loaded, so the session must restart once
# after installing. This cell does that automatically: after the restart, just "Run all" again.
import os
MARKER = "/content/.meeting_assistant_installed"
if not os.path.exists(MARKER):
    !apt-get -qq install -y ffmpeg > /dev/null
    !pip install -q whisperx "gradio>=5,<6" langchain-google-genai langchain-core google-genai openai pydantic
    open(MARKER, "w").write("ok")
    print("Install complete. Restarting the session once to load the new packages...")
    print("When it reconnects, click Runtime -> Run all again.")
    os.kill(os.getpid(), 9)
else:
    print("Packages already installed. Continuing.")

Packages already installed. Continuing.


In [ ]:
# ============================================================
# STEP 2 — CONFIG: API KEYS AND MODELS
# ============================================================
# ⚠️ Keys are written here so the notebook runs as-is. Remove them before sharing this notebook,
#    or leave them blank and add HF_TOKEN / GEMINI_API_KEY / OPENROUTER_API_KEY in Colab Secrets
#    (🔑 icon on the left) instead.
import os
from types import SimpleNamespace

GEMINI_API_KEY = "" #Put your GEMINI API KEY IN THE QUOTES
HF_TOKEN       = "" #Put your GEMINI API KEY IN THE QUOTES
OPENROUTER_API_KEY = "" #Put your OPENROUTER API KEY IN THE QUOTES

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

GEMINI_API_KEY = GEMINI_API_KEY or _secret("GEMINI_API_KEY") or _secret("GOOGLE_API_KEY") or ""
HF_TOKEN = HF_TOKEN or _secret("HF_TOKEN") or ""
OPENROUTER_API_KEY = OPENROUTER_API_KEY or _secret("OPENROUTER_API_KEY") or ""

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY      # read by langchain-google-genai

def _check_hf_token(token):
    # An invalid/revoked HF token makes even public model downloads fail with 401, so drop it.
    if not token:
        return ""
    try:
        from huggingface_hub import whoami
        print("Hugging Face account:", whoami(token=token).get("name"))
        return token
    except Exception as e:
        if "401" in str(e) or "Invalid" in str(e):
            print("⚠️ HF_TOKEN was rejected by Hugging Face (invalid or revoked). Continuing without it:\n"
                  "   speech-to-text still works, but speakers will not be labelled.\n"
                  "   Create a new token at https://huggingface.co/settings/tokens and paste it above.")
            return ""
        return token            # network hiccup: keep the token

HF_TOKEN = _check_hf_token(HF_TOKEN)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    os.environ.pop("HF_TOKEN", None)

config = SimpleNamespace(
    HF_TOKEN=HF_TOKEN,
    GOOGLE_API_KEY=GEMINI_API_KEY,
    OPENROUTER_API_KEY=OPENROUTER_API_KEY,
    # Stage 1
    WHISPER_MODEL="large-v3",    # "medium" if the GPU runs out of memory
    WHISPER_BATCH_SIZE=16,
    # Stage 2 (LLM 1)
    REFINE_MODEL="gemini-3.1-flash-lite",
    REFINE_CHUNK_CHARS=40_000,
    # Stage 3 (LLM 2): Qwen through OpenRouter, 10-minute chunks with 30 s overlap
    LLM2_MODEL="qwen/qwen-2.5-72b-instruct",
    LLM2_TOKENIZER="Qwen/Qwen2.5-72B-Instruct",   # only measures context size
    FINAL_INPUT_TOKEN_LIMIT=100_000,
    REDUCTION_BATCH_TOKEN_LIMIT=50_000,
)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch the runtime to a GPU)")
print("Gemini key set:", bool(GEMINI_API_KEY), "| HF token set:", bool(HF_TOKEN),
      "| OpenRouter key set:", bool(OPENROUTER_API_KEY))
if not OPENROUTER_API_KEY:
    print("⚠️ OPENROUTER_API_KEY is empty: stages 1 and 2 will run, but stage 3 (Qwen) will fail until you add it.")

Hugging Face account: sekuseku11
GPU: Tesla T4
Gemini key set: True | HF token set: True | OpenRouter key set: True


## Stage 1 — Speech-to-text (WhisperX)
Validation of the uploaded file, transcription, word alignment, speaker diarization, and building speaker turns.

In [3]:
# ============================================================
# STEP 3 — STAGE 1: SPEECH-TO-TEXT
# ============================================================
"""
Stage 1 of the meeting assistant: audio file -> speaker-labelled, timestamped raw transcript.
Speech-to-text model: WhisperX (Whisper large-v3) + wav2vec2 alignment + pyannote diarization.

    result = transcribe_audio("meeting.wav", context="Participants: ...")
    result["timed"]    # "[00:55 - 01:17] SPEAKER_03: text ..."
    result["plain"]    # "[SPEAKER_03] text ..."          (used by eval_transcript.py)
    result["records"]  # [{"speaker","start","end","text"}, ...]   (machine-readable)
"""
import gc, json, os, re, subprocess
from dataclasses import replace

# ----------------------------------------------------------------- errors / validation
ALLOWED_EXT = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".wma", ".mp4", ".webm", ".mkv", ".mov"}
MAX_MB = 500

class AudioError(Exception):
    """Raised with a user-friendly message when the audio cannot be processed."""

def validate_audio(path, min_seconds=1.0):
    """Check extension, size and decodability. Returns duration in seconds or raises AudioError."""
    if not path or not os.path.exists(path):
        raise AudioError("No file was provided.")
    ext = os.path.splitext(path)[1].lower()
    if ext not in ALLOWED_EXT:
        raise AudioError(f"Unsupported file type '{ext or 'none'}'. Please upload one of: "
                         + ", ".join(sorted(e.lstrip('.') for e in ALLOWED_EXT)) + ".")
    size = os.path.getsize(path)
    if size == 0:
        raise AudioError("The file is empty (0 bytes).")
    if size > MAX_MB * 1024 * 1024:
        raise AudioError(f"The file is too large ({size/1e6:.0f} MB). Maximum is {MAX_MB} MB.")
    try:
        out = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a:0", "-show_entries", "format=duration",
             "-of", "json", path], capture_output=True, text=True, timeout=60)
    except FileNotFoundError:
        raise AudioError("ffmpeg/ffprobe is not installed on the server "
                         "(install ffmpeg, e.g. `apt-get install ffmpeg` or `brew install ffmpeg`).")
    except subprocess.TimeoutExpired:
        raise AudioError("Timed out while reading the file. It may be corrupted.")
    if out.returncode != 0:
        why = (out.stderr or "").strip().splitlines()
        raise AudioError("The file could not be read as audio. It may be corrupted or not a real audio file."
                         + (f" (ffprobe: {why[-1].replace(path, os.path.basename(path))[:200]})" if why else ""))
    try:
        duration = float(json.loads(out.stdout).get("format", {}).get("duration"))
    except (TypeError, ValueError):
        # Some valid files (e.g. streamed WAVs) have no duration in the header: try decoding 1 second instead.
        test = subprocess.run(["ffmpeg", "-v", "error", "-nostdin", "-t", "1", "-i", path, "-f", "null", "-"],
                              capture_output=True, text=True, timeout=60)
        if test.returncode != 0 or test.stderr.strip():
            raise AudioError("No readable audio track was found in this file.")
        return None                                   # valid, but length unknown
    if duration < min_seconds:
        raise AudioError(f"The audio is too short ({duration:.1f}s) to transcribe.")
    return duration

# ----------------------------------------------------------------- turn building
TERM = re.compile(r"[.?!…]+[\"')]*$")
is_end = lambda w: bool(TERM.search(w["w"]))

def build_turns(result, min_words=2):
    """Group aligned words into speaker turns: [[speaker, [word dicts]], ...]
    word dict: w=text, s/e=word start/end (None if unaligned), ss/se=segment start/end (fallback)."""
    turns = []
    for seg in result["segments"]:
        last = seg.get("speaker", "UNKNOWN")
        ss, se = seg.get("start"), seg.get("end")
        words = seg.get("words") or [{"word": seg["text"].strip(), "speaker": last}]
        for w in words:
            spk = w.get("speaker", last)             # unaligned tokens inherit previous speaker
            last = spk
            item = {"w": w["word"], "s": w.get("start"), "e": w.get("end"), "ss": ss, "se": se}
            if turns and turns[-1][0] == spk: turns[-1][1].append(item)
            else: turns.append([spk, [item]])
    i = 1                                             # smooth 1-word flickers: A B A -> A
    while i < len(turns) - 1:
        if len(turns[i][1]) < min_words and turns[i-1][0] == turns[i+1][0]:
            turns[i-1][1] += turns[i][1] + turns[i+1][1]
            del turns[i:i+2]
        else:
            i += 1
    return turns

def merge_same(turns):
    out = []
    for s, w in turns:
        if not w: continue
        if out and out[-1][0] == s: out[-1][1] += w
        else: out.append([s, list(w)])
    return out

def snap_boundaries(turns, max_move=3):
    """Fix 1-3 word speaker flips that happen mid-sentence."""
    turns = merge_same(turns); i = 0
    while i < len(turns) - 1:
        p, n = turns[i][1], turns[i+1][1]; moved = False
        if p and n and not is_end(p[-1]):
            k = max([j for j, w in enumerate(p) if is_end(w)], default=-1)
            F = p[k+1:]
            c = next((j for j, w in enumerate(n) if is_end(w)), None)
            C = n[:c+1] if c is not None else None
            if C is not None and min(len(F), len(C)) <= max_move:
                if len(C) <= len(F): turns[i][1] = p + C; turns[i+1][1] = n[len(C):]
                else:                turns[i][1] = p[:k+1]; turns[i+1][1] = F + n
                moved = True
        if moved: turns = merge_same(turns); i = max(0, i-1)
        else: i += 1
    return turns

def fmt_time(t):
    t = max(0, int(round(t)))
    h, rem = divmod(t, 3600); m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def to_records(turns):
    recs = []
    for spk, words in turns:
        starts = [w["s"] for w in words if w["s"] is not None]
        ends   = [w["e"] for w in words if w["e"] is not None]
        start = min(starts) if starts else next((w["ss"] for w in words if w["ss"] is not None), 0.0)
        end   = max(ends)   if ends   else next((w["se"] for w in reversed(words) if w["se"] is not None), start)
        recs.append({"speaker": spk, "start": round(start, 2), "end": round(end, 2),
                     "text": " ".join(w["w"] for w in words).strip()})
    return recs

def format_timed(recs):
    return "\n".join(f"[{fmt_time(r['start'])} - {fmt_time(r['end'])}] {r['speaker']}: {r['text']}" for r in recs)

def format_plain(recs):
    return "\n".join(f"[{r['speaker']}] {r['text']}" for r in recs)

# ----------------------------------------------------------------- models (cached between runs)
_CACHE = {}

def get_hf_token():
    return os.environ.get("HF_TOKEN") or None

def _device():
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    return dev, ("float16" if dev == "cuda" else "int8")

def clear_models():
    """Free GPU memory (e.g. if you hit out-of-memory errors)."""
    _CACHE.clear(); gc.collect()
    try:
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception:
        pass

def _drop_bad_hf_token():
    """Hugging Face answers 401 'Repository Not Found' even for public models when the token sent is
    invalid or revoked. Remove it so public downloads (Whisper, alignment) work anonymously."""
    for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        os.environ.pop(k, None)


def _is_auth_error(e):
    msg = str(e)
    return "401" in msg or "Invalid credentials" in msg or "Invalid user token" in msg


# ----------------------------------------------------------------- main entry point
def transcribe_audio(path, context="", num_speakers=None, min_speakers=2, max_speakers=8,
                     model_size="large-v3", hf_token=None, batch_size=16, progress=print):
    """Audio file -> {'records','timed','plain','duration','diarized','warnings'}.
    Raises AudioError (with a user-friendly message) for unusable audio."""
    import whisperx
    warnings = []
    def note(msg): warnings.append(msg); progress(f"⚠️  {msg}")

    progress("1/5  Checking the audio file...")
    duration = validate_audio(path)
    device, ctype = _device()

    progress(f"2/5  Transcribing ({model_size} on {device}) — this takes a while...")
    key = ("asr", model_size, device, ctype)
    if key not in _CACHE:
        try:
            _CACHE[key] = whisperx.load_model(model_size, device, compute_type=ctype, language="en")
        except Exception as e:
            if not _is_auth_error(e):
                raise
            _drop_bad_hf_token()
            hf_token = None
            note("Hugging Face rejected HF_TOKEN (invalid or revoked), so it was ignored. "
                 "Transcription continues, but speakers cannot be labelled until you add a valid token.")
            _CACHE[key] = whisperx.load_model(model_size, device, compute_type=ctype, language="en")
    model = _CACHE[key]
    model.options = replace(model.options, initial_prompt=(context.strip() or None))  # per-recording context
    try:
        audio = whisperx.load_audio(path)
    except Exception as e:
        raise AudioError(f"The audio could not be decoded: {str(e).strip()[:300]}")
    result = model.transcribe(audio, batch_size=batch_size)
    if not result.get("segments"):
        raise AudioError("No speech was detected in this recording.")

    progress("3/5  Aligning words to timestamps...")
    try:
        akey = ("align", device)
        if akey not in _CACHE:
            _CACHE[akey] = whisperx.load_align_model(language_code="en", device=device)
        align_model, metadata = _CACHE[akey]
        result = whisperx.align(result["segments"], align_model, metadata, audio, device)
    except Exception as e:
        note(f"Word alignment failed ({type(e).__name__}); timestamps will be segment-level.")

    progress("4/5  Identifying speakers...")
    diarized = False
    token = hf_token or get_hf_token()
    try:
        if not token:
            raise RuntimeError("no valid Hugging Face token (set HF_TOKEN in the config cell or backend/.env)")
        from whisperx.diarize import DiarizationPipeline
        dkey = ("diar", device)
        if dkey not in _CACHE:
            _CACHE[dkey] = DiarizationPipeline(token=token, device=device)
        kw = {"num_speakers": num_speakers} if num_speakers else {"min_speakers": min_speakers, "max_speakers": max_speakers}
        segs = _CACHE[dkey](audio, **kw)
        result = whisperx.assign_word_speakers(segs, result, fill_nearest=True)
        diarized = True
    except Exception as e:
        note(f"Speaker identification skipped: {e}. Output will not have speaker labels.")

    progress("5/5  Building the transcript...")
    if diarized:
        records = to_records(snap_boundaries(build_turns(result)))
    else:   # no speakers: one record per segment
        records = [{"speaker": "UNKNOWN", "start": round(s.get("start", 0.0), 2), "end": round(s.get("end", 0.0), 2),
                    "text": s["text"].strip()} for s in result["segments"]]
    return {"records": records, "timed": format_timed(records), "plain": format_plain(records),
            "duration": duration, "diarized": diarized, "warnings": warnings}

print("Stage 1 ready.")

Stage 1 ready.


## Stage 2 — Transcript refinement (LLM 1)
The refinement prompt (tightened version of the LLM #1 notebook prompt). It also uses the meeting context typed in the app as evidence for names and terms. Long transcripts are refined in pieces of whole lines.

In [4]:
# ============================================================
# STEP 4 — STAGE 2: TRANSCRIPT REFINEMENT (LLM 1)
# ============================================================
"""
Stage 2 of the meeting assistant (LLM 1): raw transcript -> refined transcript.
Model: Gemini (REFINE_MODEL) via langchain-google-genai. The prompt is a tightened version of the
one from the "LLM #1" notebook (same rules, matched to the real transcript format).
"""
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


SYSTEM_PROMPT = """You are a transcript correction editor. You receive a raw meeting transcript produced by speech recognition. Return the same transcript with recognition errors corrected and made readable. You are fixing what the speech recognizer got wrong, not rewriting what people said.

PRIORITY ORDER (when rules conflict, the earlier one wins)
1. Never add information that is not in the transcript.
2. Keep the structure: every line, timestamp, speaker label and its order.
3. Keep meaning exactly, especially names, numbers and negation.
4. Correct clear recognition errors.
5. Improve readability: punctuation, capitalization, fillers, accidental repeats.

1. STRUCTURE (never change)
- One output line per input line, in the same order. Never merge, split, drop, reorder or add lines.
- Copy each line's timestamp and speaker label character for character, e.g. "[00:55 - 01:17] SPEAKER_03:". Never reformat, rename or reassign them, even if a different speaker seems more likely.

2. WHAT TO CORRECT
Correct a word or phrase only when the transcript itself gives strong evidence for the intended wording:
- Homophones and sound-alike words that do not fit the context: "let's rebound the tape" -> "let's rewind the tape"; "send it to the clime" -> "send it to the client" when a client is being discussed; "user inner face expert" -> "user interface expert".
- Names, product names, technical terms and acronyms that are spelled correctly elsewhere in the transcript or in the MEETING CONTEXT: if "Craig" is introduced and later appears as "Greg" for the same person, use "Craig".
- Obvious missing or garbled small words, punctuation and capitalization.
Read across nearby lines, not one line at a time: a normal-looking word can still be a recognition error, and an unusual word can be correct jargon.

3. WHAT TO PRESERVE EXACTLY
- Negation and polarity: "not", "no", "never", "don't", "can't", "won't", "isn't". Never add, drop or flip one.
- Numbers, amounts, prices, units, dates, times, deadlines and percentages. Do not convert, round or reformat in a way that changes meaning ("twelve fifty" may stay as spoken or become "12.50" only if clearly a price).
- Hedges and certainty: "maybe", "I think", "probably", "we could". Do not turn a suggestion into a decision or a question into a statement.
- Commitments and who made them ("I'll do it", "Craig will send it").
- Meaningful short replies: "Yes.", "No.", "Okay.", "Right.", "Sure.", "Exactly."
- The speaker's own wording and tone. Do not summarize, paraphrase, shorten or make anyone sound more formal.

4. WHAT TO REMOVE
- Fillers with no meaning: "um", "uh", "er", "erm", "you know" (only when it is filler).
- Accidental stutters and repeats: "is, is review" -> "is review". Keep repetition that is deliberate or emphatic.

5. WHEN UNSURE
If the intended word is not clearly supported by the transcript or the MEETING CONTEXT, keep the original wording. A slightly wrong-sounding original is better than a confident guess. Never use outside knowledge to fill gaps, and never invent names, numbers, facts, decisions or tasks.

OUTPUT
Return only the corrected transcript lines, in the input format:
[start - end] SPEAKER_XX: corrected speech
No headings, notes, explanations, Markdown or code fences.
"""

HUMAN_PROMPT = """MEETING CONTEXT (names, terms and topics supplied by the user; use only as evidence for spelling and terminology, never as content to add):
{context}

RAW TRANSCRIPT ({line_count} lines; return exactly {line_count} lines):
{transcript}
"""

transcript_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", HUMAN_PROMPT),
])

_refiner = None


def get_refiner():
    global _refiner
    if _refiner is None:
        if not config.GOOGLE_API_KEY:
            raise RuntimeError("GOOGLE_API_KEY is not set in backend/.env, so the refinement stage cannot run.")
        llm = ChatGoogleGenerativeAI(model=config.REFINE_MODEL, temperature=0)
        _refiner = transcript_prompt | llm
    return _refiner


def _content_text(content):
    """Gemini responses come back either as a string or as a list of content parts."""
    if isinstance(content, str):
        return content
    parts = []
    for p in content or []:
        if isinstance(p, str):
            parts.append(p)
        elif isinstance(p, dict) and p.get("type", "text") == "text":
            parts.append(p.get("text", ""))
    return "".join(parts)


def split_lines(text, max_chars):
    """Split on line boundaries into pieces of at most ~max_chars characters."""
    pieces, cur, size = [], [], 0
    for line in text.splitlines():
        if cur and size + len(line) + 1 > max_chars:
            pieces.append("\n".join(cur)); cur, size = [], 0
        cur.append(line); size += len(line) + 1
    if cur:
        pieces.append("\n".join(cur))
    return pieces


def refine_transcript(transcript, context="", progress=print):
    if not transcript or not transcript.strip():
        raise ValueError("The raw transcript is empty, so there is nothing to refine.")
    refiner = get_refiner()
    pieces = split_lines(transcript.strip(), config.REFINE_CHUNK_CHARS)
    out = []
    for i, piece in enumerate(pieces, 1):
        progress(f"Refining transcript with {config.REFINE_MODEL}"
                 + (f" (part {i}/{len(pieces)})" if len(pieces) > 1 else "") + "...")
        response = refiner.invoke({
            "transcript": piece,
            "context": context.strip() or "(none provided)",
            "line_count": len(piece.splitlines()),
        })
        text = "\n".join(l for l in _content_text(response.content).strip().splitlines() if l.strip())
        if not text:
            raise RuntimeError(f"{config.REFINE_MODEL} returned an empty refined transcript"
                               + (f" for part {i}." if len(pieces) > 1 else "."))
        out.append(text)
    return "\n".join(out)

print("Stage 2 ready.")

Stage 2 ready.


## Stage 3 — Minutes, decisions and action items (LLM 2)
Part 7 of the Qwen LLM2 notebook: parse the timestamps → 10-minute time windows with 30 s overlap → per-window analysis with **Qwen 2.5 72B** (OpenRouter) → condense if the analyses exceed 100k tokens → final JSON → Pydantic validation.

In [5]:
# ============================================================
# STEP 5 — STAGE 3: MEETING RECORD (LLM 2)
# ============================================================
"""
Stage 3 of the meeting assistant (LLM 2): refined transcript -> minutes, decisions, action items.

Follows Part 7 of inter_ml_3_llm2_qwen-2.0:
  parse the timestamped transcript -> 10-minute time chunks with 30-second overlap ->
  per-chunk analysis with Qwen 2.5 72B (OpenRouter) -> combine -> condense if the analyses
  exceed 100k tokens (Qwen tokenizer, used only to measure size) -> final JSON -> Pydantic.
"""
import json
import re

from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError


CHUNK_DURATION_SECONDS = 10 * 60
CHUNK_OVERLAP_SECONDS = 30
CHUNK_STEP_SECONDS = CHUNK_DURATION_SECONDS - CHUNK_OVERLAP_SECONDS


# ============================================================
# PYDANTIC OUTPUT SCHEMA
# ============================================================

class ActionItem(BaseModel):
    action: str
    assigned_to: str | None = None
    deadline: str | None = None


class LLM2Output(BaseModel):
    minutes_of_meeting: str
    decisions_made: list[str] = Field(default_factory=list)
    action_items: list[ActionItem] = Field(default_factory=list)


# ============================================================
# LLM CALL (Qwen 2.5 72B through OpenRouter)
# ============================================================

_client = None


def _get_client():
    global _client
    if _client is None:
        if not config.OPENROUTER_API_KEY:
            raise RuntimeError("OPENROUTER_API_KEY is not set, so the minutes stage (Qwen via OpenRouter) cannot run.")
        _client = OpenAI(api_key=config.OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
    return _client


def generate_llm_response(prompt):
    if not isinstance(prompt, str):
        raise TypeError("Prompt must be a string.")
    if not prompt.strip():
        raise ValueError("Prompt cannot be empty.")
    response = _get_client().chat.completions.create(
        model=config.LLM2_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    if not response.choices:
        raise ValueError("Qwen returned no choices.")
    text = response.choices[0].message.content
    if not text or not text.strip():
        raise ValueError("Qwen returned an empty response.")
    return text


# ============================================================
# TOKENIZER (only measures context size)
# ============================================================

_tokenizer = None


class _ApproxTokenizer:
    """Fallback when the Qwen tokenizer cannot be downloaded: ~4 characters per token."""

    def encode(self, text, add_special_tokens=False):
        return range(0, len(text), 4)


def _get_tokenizer():
    global _tokenizer
    if _tokenizer is None:
        try:
            from transformers import AutoTokenizer
            _tokenizer = AutoTokenizer.from_pretrained(config.LLM2_TOKENIZER, trust_remote_code=True)
        except Exception as e:
            print(f"Could not load tokenizer {config.LLM2_TOKENIZER} ({e}); using a ~4 chars/token estimate.")
            _tokenizer = _ApproxTokenizer()
    return _tokenizer


def count_text_tokens(text):
    if not text:
        return 0
    return len(_get_tokenizer().encode(text, add_special_tokens=False))


# ============================================================
# TIMESTAMP PARSING AND TIME-BASED CHUNKING
# ============================================================

def timestamp_to_seconds(timestamp):
    """MM:SS, HH:MM:SS, MM:SS.mmm or HH:MM:SS.mmm -> seconds."""
    parts = timestamp.replace(",", ".").split(":")
    if len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    raise ValueError(f"Invalid timestamp format: {timestamp}")


def seconds_to_timestamp(seconds):
    seconds = max(0, int(round(seconds)))
    return f"{seconds // 3600:02d}:{seconds % 3600 // 60:02d}:{seconds % 60:02d}"


_TIMESTAMP_LINE = re.compile(
    r"""
    ^\s*
    \[
    (\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d+)?)
    \s*[-–—]\s*
    (\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d+)?)
    \]
    \s*
    (.*)
    $
    """,
    re.VERBOSE,
)


def parse_timestamped_transcript(transcript):
    """'[00:14 - 00:30] SPEAKER_01: text' lines -> [{start, end, text}]. Wrapped lines are joined."""
    if not isinstance(transcript, str):
        raise TypeError("Transcript must be a string.")
    if not transcript.strip():
        raise ValueError("Transcript is empty.")
    segments, start, end, text = [], None, None, []

    def flush():
        joined = " ".join(text).strip()
        if start is not None and joined:
            segments.append({"start": start, "end": end, "text": joined})

    for line in transcript.splitlines():
        m = _TIMESTAMP_LINE.match(line)
        if m:
            flush()
            start, end = timestamp_to_seconds(m.group(1)), timestamp_to_seconds(m.group(2))
            text = [m.group(3).strip()] if m.group(3).strip() else []
        elif line.strip() and start is not None:
            text.append(line.strip())
    flush()
    if not segments:
        raise ValueError("No timestamped transcript segments were found. "
                         "Expected format such as [00:14 - 00:30] SPEAKER_01: text")
    segments.sort(key=lambda s: s["start"])
    return segments


def create_time_chunks(segments):
    """10-minute windows with 30-second overlap; a segment goes in every window it overlaps."""
    if not segments:
        raise ValueError("No transcript segments available.")
    recording_end = max(s["end"] for s in segments)
    chunks, chunk_start, chunk_id = [], 0.0, 1
    while chunk_start < recording_end:
        chunk_end = min(chunk_start + CHUNK_DURATION_SECONDS, recording_end)
        lines = [
            f"[{seconds_to_timestamp(s['start'])} - {seconds_to_timestamp(s['end'])}] {s['text']}"
            for s in segments
            if s["end"] > chunk_start and s["start"] < chunk_end
        ]
        chunk_text = "\n".join(lines).strip()
        if chunk_text:
            chunks.append({"chunk_id": chunk_id, "start_time": chunk_start, "end_time": chunk_end,
                           "duration_seconds": chunk_end - chunk_start, "text": chunk_text})
            chunk_id += 1
        chunk_start += CHUNK_STEP_SECONDS
    if not chunks:
        raise ValueError("No chunks could be created from transcript.")
    return chunks


def prepare_transcript_chunks(refined_transcript, progress=print):
    try:
        segments = parse_timestamped_transcript(refined_transcript)
    except ValueError:
        if not refined_transcript or not refined_transcript.strip():
            raise
        # Timestamps were lost somewhere: analyse the whole transcript as one section.
        progress("No timestamps found in the refined transcript; analysing it as one section.")
        return {"segments": [], "total_duration": 0.0,
                "chunks": [{"chunk_id": 1, "start_time": 0.0, "end_time": 0.0,
                            "duration_seconds": 0.0, "text": refined_transcript.strip()}]}
    chunks = create_time_chunks(segments)
    total = max(s["end"] for s in segments)
    progress(f"Parsed {len(segments)} segments (~{seconds_to_timestamp(total)}) into "
             f"{len(chunks)} time chunk(s) of 10 min with 30 s overlap.")
    return {"segments": segments, "chunks": chunks, "total_duration": total}


# ============================================================
# PROMPTS (verbatim from Part 7)
# ============================================================

CHUNK_ANALYSIS_PROMPT = """
You are analyzing one time section of a meeting transcript.

This section belongs to a larger meeting and may overlap
slightly with another section.

Your job is to extract information that will later be used
to create:

1. Minutes of Meeting
2. Decisions Made
3. Key Action Items

IMPORTANT:
Do NOT summarize vaguely. Explicitly identify decisions
and action items whenever they occur.

============================================================
1. IMPORTANT DISCUSSION
============================================================

Capture the important topics, discussions, requirements,
ideas, problems, questions, and conclusions discussed in
this section.

============================================================
2. DECISIONS MADE
============================================================

Look very carefully for statements where the participants
actually make a decision.

A decision includes things that are:

- agreed upon
- approved
- rejected
- finalized
- confirmed
- selected
- chosen
- accepted
- explicitly decided

Decision examples:

"Let's go with option A."
"Okay, we've agreed to use the original design."
"We'll use Python for this."
"That's finalized."
"Yes, let's proceed with that."
"Everyone agrees that we should use this approach."

IMPORTANT:

Do NOT report something as a decision merely because it was
discussed, suggested, proposed, considered, or recommended.

For every decision, preserve the actual meaning of what
was decided.

============================================================
3. KEY ACTION ITEMS
============================================================

Only report an action item if someone explicitly:

- agrees to do something
- is assigned something
- commits to doing something
- is told to do something
- states that they will do something

Do NOT treat these as action items:

- suggestions
- ideas
- opinions
- questions
- general discussion
- information
- hypothetical tasks
- past actions
- things that might be done later but were never agreed upon

If the same action appears multiple times in this section,
report it only once.

============================================================
4. RESPONSIBILITY
============================================================

If a person is explicitly assigned an action, record their
name.

If no person is explicitly assigned, do not invent one.

============================================================
5. DEADLINE
============================================================

If a deadline is explicitly mentioned, record it.

If there is no explicit deadline, do not invent one.

============================================================
IMPORTANT RULES
============================================================

Only use information explicitly present in the transcript.

Do NOT invent:

- decisions
- action items
- names
- responsibilities
- deadlines
- facts

Do NOT convert suggestions into decisions.

Do NOT convert discussion into decisions.

Do NOT convert discussion into action items.

Do NOT produce the final JSON.

Return a concise analysis containing these sections:

IMPORTANT DISCUSSION:
...

DECISIONS MADE:
- ...

ACTION ITEMS:
- ...

Transcript section:

{chunk}
"""


REDUCTION_PROMPT = """
You are consolidating analyses from multiple sections of the
same meeting transcript.

The sections may overlap because the transcript was divided
into overlapping time windows.

Create one consolidated meeting analysis.

Your task is to combine the information accurately, remove
duplicates, and preserve important information.

IMPORTANT DISCUSSION:
Combine important discussion points while removing
unnecessary repetition.

Do not remove important context.

DECISIONS:
Preserve decisions that are explicitly supported by the
provided analyses.

Do not create new decisions.

Do not turn suggestions, proposals, opinions, or discussions
into decisions.

If the same decision appears in multiple overlapping
sections, merge it into one decision while preserving its
meaning.

ACTION ITEMS:
Preserve only action items that are explicitly supported by
the provided analyses.

An action should represent:
1. A concrete task.
2. An explicit commitment or assignment to perform that task.

Do not create new action items.

Do not infer tasks from suggestions, discussions, information,
or implied responsibilities.

If the same action appears multiple times because of overlap,
merge it into one action.

RESPONSIBILITY AND DEADLINE:
Preserve explicitly stated responsibilities and deadlines.

Do not infer missing information.

STRICT ACCURACY:
- Use only information present in the provided analyses.
- Do not invent facts.
- Do not invent decisions.
- Do not invent action items.
- Do not invent names.
- Do not invent responsibilities.
- Do not invent deadlines.
- Do not reinterpret the meeting.
- Do not upgrade a suggestion into a decision.
- Do not upgrade discussion into an action item.
- Merge duplicates caused by overlapping sections.

The purpose of this stage is consolidation, not additional
interpretation.

Return concise plain text using:

IMPORTANT DISCUSSION:
- ...

DECISIONS MADE:
- ...

ACTION ITEMS:
- ...

CONSOLIDATED ANALYSIS:
...
"""


FINAL_OUTPUT_PROMPT = """
You are creating the final structured output of a meeting.

Using the consolidated meeting analysis below, produce:

1. Minutes of Meeting
2. Decisions Made
3. Key Actions

MINUTES OF MEETING:
Summarize the important discussions, topics, requirements,
ideas, and conclusions accurately.

Do not convert general discussion into decisions or actions.

DECISIONS MADE:
Include only decisions that are explicitly supported by the
meeting analysis.

A decision should represent something that was actually
decided, agreed, approved, rejected, finalized, confirmed,
selected, or otherwise settled.

Do not classify suggestions, proposals, opinions, questions,
possibilities, or general discussion as decisions.

If there is insufficient evidence that a decision was made,
do not include it.

KEY ACTIONS:
Include only genuine action items.

An action item must contain:
1. A concrete task or activity.
2. An explicit commitment, agreement, or assignment to
   perform that task.

Do not create action items from suggestions, ideas, opinions,
questions, general discussion, information, or implied
responsibilities.

Do not infer that someone is responsible for a task simply
because they discussed it.

Preserve the actual meaning of each task.

RESPONSIBILITY:
Use the person's actual name when their identity is explicitly
established in the transcript.

Do not guess identities.

If responsibility is not explicitly known, use null.

DEADLINE:
Include a deadline only when explicitly stated.

Do not infer deadlines.

If no deadline is explicitly stated, use null.

DUPLICATES:
The source transcript was processed using overlapping
sections.

Merge duplicate decisions and duplicate action items.

Each distinct decision or action should appear only once.

FINAL ACCURACY RULE:
Use only information supported by the meeting analysis.

Do not invent:
- decisions
- action items
- names
- responsibilities
- deadlines
- facts

Do not turn suggestions into decisions.

Do not turn discussion into action items.

When uncertain, prefer omitting an unsupported decision or
action rather than making an inference.

Return ONLY valid JSON.

Use exactly this structure:

{
  "minutes_of_meeting": "string",
  "decisions_made": [
    "decision 1"
  ],
  "action_items": [
    {
      "action": "specific task",
      "assigned_to": "person or null",
      "deadline": "deadline or null"
    }
  ]
}

If there are no explicit decisions:
"decisions_made": []

If there are no explicit action items:
"action_items": []

Consolidated meeting analysis:

{combined_analyses}
"""


# ============================================================
# PIPELINE
# ============================================================

def process_single_chunk(chunk):
    start, end = seconds_to_timestamp(chunk["start_time"]), seconds_to_timestamp(chunk["end_time"])
    prompt = CHUNK_ANALYSIS_PROMPT.replace("{chunk}", chunk["text"])
    return generate_llm_response(f"TIME WINDOW: {start} → {end}\n\n" + prompt)


def process_all_chunks(chunked_transcript, progress=print):
    chunks = chunked_transcript["chunks"]
    analyses = []
    for chunk in chunks:
        cid = chunk["chunk_id"]
        start, end = seconds_to_timestamp(chunk["start_time"]), seconds_to_timestamp(chunk["end_time"])
        progress(f"Analysing chunk {cid}/{len(chunks)} ({start} → {end}) with {config.LLM2_MODEL}...")
        try:
            analysis = process_single_chunk(chunk)
        except Exception as e:
            raise RuntimeError(f"{config.LLM2_MODEL} failed on chunk {cid}: {e}") from e
        if analysis and analysis.strip():
            analyses.append({"chunk_id": cid, "start_time": start, "end_time": end, "analysis": analysis})
    if not analyses:
        raise ValueError("No successful chunk analyses were produced.")
    return analyses


def combine_chunk_outputs(analyses):
    return "\n\n".join(
        f"===== CHUNK {a['chunk_id']} =====\nTime: {a['start_time']} → {a['end_time']}\n\n{a['analysis']}"
        for a in analyses
    )


def reduce_analysis_batch(analysis_batch):
    return generate_llm_response(REDUCTION_PROMPT + "\n\nANALYSES:\n\n" + "\n\n".join(analysis_batch))


def make_context_safe(combined_analyses, progress=print):
    token_count = count_text_tokens(combined_analyses)
    if token_count <= config.FINAL_INPUT_TOKEN_LIMIT:
        return combined_analyses
    progress(f"Combined analysis is {token_count:,} tokens; condensing before the final call...")
    batches, current, current_tokens = [], [], 0
    for section in combined_analyses.split("\n\n===== CHUNK "):
        section = section.strip()
        if not section:
            continue
        n = count_text_tokens(section)
        if current and current_tokens + n > config.REDUCTION_BATCH_TOKEN_LIMIT:
            batches.append("\n\n===== CHUNK ".join(current))
            current, current_tokens = [], 0
        current.append(section)
        current_tokens += n
    if current:
        batches.append("\n\n===== CHUNK ".join(current))
    reduced = []
    for i, batch in enumerate(batches, 1):
        progress(f"Condensing batch {i}/{len(batches)}...")
        reduced.append(reduce_analysis_batch([batch]))
    return "\n\n".join(reduced)


def validate_final_output(raw_output):
    if not raw_output:
        raise ValueError("Final Qwen output is empty.")
    cleaned = raw_output.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):]
    elif cleaned.startswith("```"):
        cleaned = cleaned[len("```"):]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-len("```")]
    cleaned = cleaned.strip()
    first, last = cleaned.find("{"), cleaned.rfind("}")
    if first == -1 or last == -1 or last <= first:
        raise ValueError("Could not find a valid JSON object in the Qwen output.")
    try:
        parsed = json.loads(cleaned[first:last + 1])
    except json.JSONDecodeError as e:
        raise ValueError(f"Qwen returned invalid JSON: {e}")
    try:
        return LLM2Output.model_validate(parsed)
    except ValidationError as e:
        raise ValueError(f"Final output failed Pydantic validation:\n{e}")


def run_llm2(refined_transcript, progress=print):
    """Refined transcript -> (validated LLM2Output, stats)."""
    chunked = prepare_transcript_chunks(refined_transcript, progress)
    analyses = process_all_chunks(chunked, progress)
    safe = make_context_safe(combine_chunk_outputs(analyses), progress)
    progress("Generating minutes, decisions and action items...")
    prompt = FINAL_OUTPUT_PROMPT.replace("{combined_analyses}", safe)
    raw = generate_llm_response(prompt)
    try:
        result = validate_final_output(raw)
    except ValueError:
        progress("Model output was not valid JSON; asking once more...")
        raw = generate_llm_response(prompt + "\n\nYour previous reply was not valid JSON. Reply with ONLY the JSON object.")
        result = validate_final_output(raw)
    stats = {"model": config.LLM2_MODEL, "segments": len(chunked["segments"]),
             "total_chunks": len(chunked["chunks"]),
             "duration": seconds_to_timestamp(chunked["total_duration"])}
    return result, stats


# ============================================================
# EXPORT (human-readable text, same content as the JSON)
# ============================================================

UNSPECIFIED = "Unspecified"


def to_text(result: dict):
    lines = ["1. MINUTES OF MEETING", "=" * 70, "", result["minutes_of_meeting"].strip(), "", "",
             "2. DECISIONS MADE", "=" * 70, ""]
    decisions = result.get("decisions_made") or []
    lines += [f"{i}. {d}" for i, d in enumerate(decisions, 1)] or ["No explicit decisions identified."]
    lines += ["", "", "3. KEY ACTIONS", "=" * 70, ""]
    items = result.get("action_items") or []
    if items:
        for i, it in enumerate(items, 1):
            lines += [f"{i}. {it['action']}",
                      f"   Assigned to: {it.get('assigned_to') or UNSPECIFIED}",
                      f"   Deadline: {it.get('deadline') or UNSPECIFIED}", ""]
    else:
        lines.append("No action items identified.")
    return "\n".join(lines).rstrip() + "\n"

print("Stage 3 ready.")

Stage 3 ready.


## Interface
Runs the three stages in order on each upload and shows per-stage status, both transcripts side by side, the meeting record, and downloads.

In [6]:
# ============================================================
# STEP 6 — PIPELINE RUNNER + GRADIO INTERFACE
# ============================================================
import difflib, html, json, queue, re, threading, time, traceback, uuid, zipfile
from datetime import datetime
from pathlib import Path
import gradio as gr

OUT_ROOT = Path("/content/meeting_outputs") if Path("/content").exists() else Path("meeting_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

STAGES = [("transcribe", "Speech-to-text", f"WhisperX {config.WHISPER_MODEL} + pyannote"),
          ("refine", "Transcript refinement", f"LLM 1 · {config.REFINE_MODEL}"),
          ("minutes", "Minutes, decisions & actions", f"LLM 2 · {config.LLM2_MODEL.split('/')[-1]}")]
SPEAKER_COLORS = ["#6366f1", "#0ea5e9", "#f59e0b", "#10b981", "#ec4899", "#8b5cf6", "#ef4444", "#14b8a6"]


def friendly_error(e):
    if isinstance(e, AudioError):
        return str(e)
    msg = str(e).strip() or type(e).__name__
    low = msg.lower()
    if "huggingface.co" in low and ("401" in low or "repository not found" in low):
        return ("Hugging Face rejected the HF_TOKEN in the config cell (it is invalid or was revoked), so the "
                "speech model could not be downloaded. Paste a new token from https://huggingface.co/settings/tokens, "
                "or leave HF_TOKEN empty to run without speaker labels, then run the cells again.")
    if "is not set" in low:
        return msg + " Add the key in the config cell (or Colab Secrets) and run the cells again."
    if "api key" in low or "api_key" in low or "permission" in low or "401" in low or "403" in low:
        return f"The model provider rejected the request. Check the API key in the config cell. ({msg[:300]})"
    if "402" in low or "credits" in low:
        return f"The OpenRouter account has no credits left for this model. ({msg[:300]})"
    if "429" in low or "quota" in low or ("rate" in low and "limit" in low):
        return f"The model provider's rate limit or quota was hit. Wait a minute and try again. ({msg[:300]})"
    if "out of memory" in low:
        return "The GPU ran out of memory. Set WHISPER_MODEL to 'medium' or lower WHISPER_BATCH_SIZE in the config cell."
    return msg[:1000]


def fmt_secs(s):
    s = int(s or 0)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s" if s >= 60 else f"{s}s"


# ------------------------------------------------------------------ HTML renderers

def blank_state():
    return {"stages": {k: {"status": "pending", "msg": ""} for k, _, _ in STAGES}, "warnings": [], "error": None,
            "done": False}


def status_html(state):
    stages = state["stages"]
    done = sum(s["status"] == "completed" for s in stages.values())
    running = any(s["status"] == "running" for s in stages.values())
    pct = 100 if state.get("done") else int((done + (0.5 if running else 0)) / len(STAGES) * 100)
    cards = []
    for i, (key, label, model) in enumerate(STAGES, 1):
        s = stages[key]
        st = s["status"]
        icon = {"pending": str(i), "running": "<span class='ma-spin'></span>", "completed": "✓",
                "failed": "✕", "skipped": "–"}[st]
        badge = {"pending": "Waiting", "running": "Running", "completed": "Done", "failed": "Failed",
                 "skipped": "Not run"}[st]
        t = f" · {fmt_secs((s.get('end') or time.time()) - s['start'])}" if s.get("start") else ""
        msg = html.escape(s.get("msg", "")) if st == "running" else ""
        if st == "skipped":
            msg = "Skipped because an earlier stage failed."
        cards.append(
            f"<div class='ma-step ma-{st}'><div class='ma-step-top'><div class='ma-dot'>{icon}</div>"
            f"<span class='ma-badge'>{badge}{t}</span></div>"
            f"<div class='ma-step-label'>Stage {i} · {label}</div><div class='ma-step-model'>{html.escape(model)}</div>"
            + (f"<div class='ma-step-msg'>{msg}</div>" if msg else "") + "</div>")
    extra = ""
    if state.get("error"):
        extra += f"<div class='ma-alert ma-error'><b>Processing failed.</b> {html.escape(state['error'])}</div>"
    for w in state.get("warnings", []):
        extra += f"<div class='ma-alert ma-warn'><b>Note.</b> {html.escape(w)}</div>"
    if state.get("done"):
        total = fmt_secs(time.time() - state["t0"]) if state.get("t0") else ""
        extra += (f"<div class='ma-alert ma-ok'><b>All stages completed{' in ' + total if total else ''}.</b> "
                  "Results and downloads are below.</div>")
    bar = "ma-bar-fail" if state.get("error") else ""
    return (f"<div class='ma-status'><div class='ma-progress {bar}'><div style='width:{pct}%'></div></div>"
            f"<div class='ma-steps'>{''.join(cards)}</div>{extra}</div>")


def message_html(msg, kind="error", title="Could not start processing."):
    return (f"<div class='ma-status'><div class='ma-alert ma-{kind}'><b>{html.escape(title)}</b> "
            f"{html.escape(msg)}</div></div>")


def overview_html(state):
    if not state.get("raw"):
        return ""
    rec = state.get("record")
    cards = [("Duration", fmt_secs(state.get("duration"))),
             ("Speakers", str(state.get("speakers") or "–")),
             ("Words corrected", str(state.get("changes", "–")) if state.get("refined") else "–"),
             ("Key decisions", str(len(rec.get("decisions_made") or [])) if rec else "–"),
             ("Action items", str(len(rec.get("action_items") or [])) if rec else "–")]
    return "<div class='ma-stats'>" + "".join(
        f"<div class='ma-stat'><div class='ma-stat-v'>{v}</div><div class='ma-stat-k'>{k}</div></div>"
        for k, v in cards) + "</div>"


_LINE = re.compile(r"^\s*\[([^\]]+)\]\s*([^:\[\]]{1,40}):\s*(.*)$")


def _speaker_badge(name, colors):
    c = colors.setdefault(name, SPEAKER_COLORS[len(colors) % len(SPEAKER_COLORS)])
    return f"<span class='ma-spk' style='--c:{c}'>{html.escape(name)}</span>"


def _words_diff(a, b):
    wa, wb = a.split(), b.split()
    norm = lambda w: re.sub(r"[^\w']", "", w.lower())
    sm = difflib.SequenceMatcher(None, [norm(w) for w in wa], [norm(w) for w in wb], autojunk=False)
    left, right, changes = [], [], 0
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        L = html.escape(" ".join(wa[i1:i2])); R = html.escape(" ".join(wb[j1:j2]))
        if op == "equal":
            left.append(L); right.append(R)
        else:
            changes += max(i2 - i1, j2 - j1)
            if L: left.append(f"<mark class='ma-del'>{L}</mark>")
            if R: right.append(f"<mark class='ma-ins'>{R}</mark>")
    return " ".join(left), " ".join(right), changes


def _line_html(line, body, colors):
    m = _LINE.match(line)
    if not m:
        return f"<div class='ma-line'><span class='ma-text'>{body}</span></div>"
    return (f"<div class='ma-line'><div class='ma-meta'><span class='ma-ts'>{html.escape(m.group(1))}</span>"
            f"{_speaker_badge(m.group(2).strip(), colors)}</div><div class='ma-text'>{body}</div></div>")


def diff_html(raw, refined):
    """Side-by-side transcripts with speaker badges; changed words highlighted on both sides.
    Returns (html, number_of_changed_words)."""
    colors, changes = {}, 0
    rl = [l for l in raw.splitlines() if l.strip()]
    fl = [l for l in (refined or "").splitlines() if l.strip()]
    if refined and len(rl) == len(fl):
        L, R = [], []
        for a, b in zip(rl, fl):
            ma, mb = _LINE.match(a), _LINE.match(b)
            ta, tb = (ma.group(3) if ma else a), (mb.group(3) if mb else b)
            da, db, n = _words_diff(ta, tb)
            changes += n
            L.append(_line_html(a, da, colors)); R.append(_line_html(b, db, colors))
        L, R = "".join(L), "".join(R)
    elif refined:
        a, b, changes = _words_diff(raw, refined)
        L, R = f"<div class='ma-line'>{a}</div>", f"<div class='ma-line'>{b}</div>"
    else:
        L = "".join(_line_html(x, html.escape(_LINE.match(x).group(3) if _LINE.match(x) else x), colors) for x in rl)
        R = "<div class='ma-empty-in'>Waiting for the refinement stage…</div>"
    return ("<div class='ma-panes'>"
            f"<div class='ma-pane'><div class='ma-pane-h'><span class='ma-pill ma-pill-raw'>Raw</span>Speech-to-text output</div>"
            f"<div class='ma-tx'>{L}</div></div>"
            f"<div class='ma-pane'><div class='ma-pane-h'><span class='ma-pill ma-pill-ref'>Refined</span>After LLM 1 correction</div>"
            f"<div class='ma-tx'>{R}</div></div></div>"), changes


def empty_html(icon, title, sub):
    return (f"<div class='ma-empty'><div class='ma-empty-icon'>{icon}</div><div class='ma-empty-t'>{title}</div>"
            f"<div class='ma-empty-s'>{sub}</div></div>")


def minutes_html(record):
    if not record:
        return empty_html("📝", "No minutes yet", "The minutes of meeting appear here when processing finishes.")
    text = record["minutes_of_meeting"].strip()
    parts = []
    for block in re.split(r"\n\s*\n", text):
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        if lines and all(re.match(r"^([-*•]|\d+[.)])\s+", l) for l in lines):
            parts.append("<ul>" + "".join(f"<li>{html.escape(re.sub(r'^([-*•]|\d+[.)])\s+', '', l))}</li>"
                                          for l in lines) + "</ul>")
        elif lines:
            parts.append(f"<p>{html.escape(' '.join(lines))}</p>")
    return f"<div class='ma-card ma-prose'>{''.join(parts)}</div>"


def decisions_html(record):
    if not record:
        return empty_html("⚖️", "No decisions yet", "Key decisions appear here when processing finishes.")
    d = record.get("decisions_made") or []
    if not d:
        return empty_html("🤝", "No decisions were reached in this meeting",
                          "The model found no explicit decision in the transcript.")
    return "<div class='ma-list'>" + "".join(
        f"<div class='ma-item'><div class='ma-num'>{i}</div><div class='ma-item-t'>{html.escape(x)}</div></div>"
        for i, x in enumerate(d, 1)) + "</div>"


def _chip(icon, value):
    if value:
        return f"<span class='ma-chip'>{icon} {html.escape(str(value))}</span>"
    return f"<span class='ma-chip ma-chip-none'>{icon} Unspecified</span>"


def actions_html(record):
    if not record:
        return empty_html("🎯", "No action items yet", "Action items appear here when processing finishes.")
    items = record.get("action_items") or []
    if not items:
        return empty_html("🗒️", "No action items were assigned in this meeting",
                          "The model found no explicit task or commitment in the transcript.")
    return "<div class='ma-list'>" + "".join(
        f"<div class='ma-item'><div class='ma-num'>{i}</div><div class='ma-item-body'>"
        f"<div class='ma-item-t'>{html.escape(a['action'])}</div>"
        f"<div class='ma-chips'>{_chip('👤', a.get('assigned_to'))}{_chip('📅', a.get('deadline'))}</div></div></div>"
        for i, a in enumerate(items, 1)) + "</div>"


def to_markdown(record, filename):
    lines = [f"# Meeting record: {filename}", "", "## Minutes of meeting", "", record["minutes_of_meeting"].strip(), "",
             "## Key decisions", ""]
    d = record.get("decisions_made") or []
    lines += [f"{i}. {x}" for i, x in enumerate(d, 1)] or ["No decisions were reached in this meeting."]
    lines += ["", "## Action items", ""]
    items = record.get("action_items") or []
    if items:
        lines += ["| # | Task | Owner | Deadline |", "|---|---|---|---|"]
        lines += [f"| {i} | {a['action']} | {a.get('assigned_to') or 'Unspecified'} | {a.get('deadline') or 'Unspecified'} |"
                  for i, a in enumerate(items, 1)]
    else:
        lines.append("No action items were assigned in this meeting.")
    return "\n".join(lines) + "\n"


# ------------------------------------------------------------------ pipeline

def run_pipeline(audio_path, context, num_speakers, state, events):
    """Runs the 3 stages in order. Progress goes to `events`; results to `state`."""
    def log(msg):
        state["stages"][state["stage"]]["msg"] = msg
        events.put("progress")

    def start(key):
        state["stage"] = key
        state["stages"][key].update(status="running", start=time.time())
        events.put("progress")

    def finish(key):
        state["stages"][key].update(status="completed", end=time.time())
        events.put("progress")

    out = state["out_dir"]
    try:
        start("transcribe")
        res = transcribe_audio(audio_path, context=context or "", num_speakers=num_speakers,
                               model_size=config.WHISPER_MODEL, hf_token=config.HF_TOKEN or None,
                               batch_size=config.WHISPER_BATCH_SIZE, progress=log)
        state["warnings"] += res["warnings"]
        state["speakers"] = len({r["speaker"] for r in res["records"]})
        state["duration"] = res.get("duration") or state["duration"]
        state["raw"] = res["timed"]
        (out / "raw_transcript.txt").write_text(res["timed"], encoding="utf-8")
        (out / "raw_transcript.json").write_text(json.dumps(res["records"], indent=2, ensure_ascii=False), encoding="utf-8")
        finish("transcribe")

        start("refine")
        state["refined"] = refine_transcript(res["timed"], context=context or "", progress=log)
        (out / "refined_transcript.txt").write_text(state["refined"], encoding="utf-8")
        finish("refine")

        start("minutes")
        result, _stats = run_llm2(state["refined"], progress=log)
        record = result.model_dump()
        state["record"] = record
        (out / "meeting_record.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
        (out / "meeting_record.txt").write_text(to_text(record), encoding="utf-8")
        (out / "meeting_record.md").write_text(to_markdown(record, state["filename"]), encoding="utf-8")
        finish("minutes")
        state["done"] = True
    except Exception as e:
        traceback.print_exc()
        key = state.get("stage")
        if key:
            state["stages"][key].update(status="failed", end=time.time())
            after = False
            for k, _, _ in STAGES:
                if after:
                    state["stages"][k]["status"] = "skipped"
                after = after or k == key
        label = {k: f"Stage {i} ({l})" for i, (k, l, _) in enumerate(STAGES, 1)}.get(key, "Processing")
        state["error"] = f"{label} failed: {friendly_error(e)}"
    finally:
        files = [p for p in sorted(out.iterdir()) if p.suffix in (".txt", ".json", ".md")]
        if files:
            zpath = out / f"{Path(state['filename']).stem}_outputs.zip"
            with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
                for p in files:
                    z.write(p, p.name)
            state["files"] = [str(zpath)] + [str(p) for p in files]
        events.put("done")


def render(state):
    rec = state.get("record")
    tx = ""
    if state.get("raw"):
        tx, state["changes"] = diff_html(state["raw"], state["refined"])
    else:
        tx = empty_html("🎙️", "No transcript yet", "The raw and refined transcripts appear here side by side.")
    return (status_html(state), overview_html(state), tx, minutes_html(rec), decisions_html(rec),
            actions_html(rec), state.get("files") or None)


def process(file_path, context, num_speakers):
    """Gradio handler: validates, then streams status and results while the pipeline runs."""
    if not file_path:
        yield (message_html("Please choose an audio file first.", "warn", "No file selected."),) + render(blank_state())[1:]
        return
    try:
        duration = validate_audio(file_path)
    except AudioError as e:
        yield (message_html(str(e)),) + render(blank_state())[1:]
        return

    name = Path(file_path).name
    run_id = datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:4]
    out = OUT_ROOT / run_id
    out.mkdir(parents=True)
    n = int(num_speakers) if num_speakers and num_speakers > 0 else None
    state = {**blank_state(), "filename": name, "out_dir": out, "stage": None, "t0": time.time(),
             "duration": duration, "raw": "", "refined": "", "record": None, "files": []}
    events = queue.Queue()
    threading.Thread(target=run_pipeline, args=(file_path, context, n, state, events), daemon=True).start()

    while True:
        try:
            ev = events.get(timeout=1.0)
        except queue.Empty:
            ev = "tick"
        yield render(state)
        if ev == "done":
            break


# ------------------------------------------------------------------ look and feel

THEME = gr.themes.Soft(
    primary_hue="indigo", secondary_hue="violet", neutral_hue="slate", radius_size="lg",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "Consolas", "monospace"],
).set(
    body_background_fill="#f5f6fb", body_background_fill_dark="#0b0e17",
    block_background_fill_dark="#121625", block_border_color_dark="#232842",
    block_shadow="0 1px 2px rgba(16,24,40,.04), 0 8px 24px rgba(16,24,40,.06)",
    block_radius="16px",
    button_primary_background_fill="linear-gradient(135deg,#6366f1 0%,#8b5cf6 55%,#d946ef 100%)",
    button_primary_background_fill_hover="linear-gradient(135deg,#4f46e5 0%,#7c3aed 55%,#c026d3 100%)",
    button_primary_background_fill_dark="linear-gradient(135deg,#6366f1 0%,#8b5cf6 55%,#d946ef 100%)",
    button_primary_background_fill_hover_dark="linear-gradient(135deg,#4f46e5 0%,#7c3aed 55%,#c026d3 100%)",
    button_primary_text_color="#ffffff", button_primary_border_color="transparent",
    button_large_radius="14px", button_large_padding="16px 24px",
)

CSS = """
.gradio-container{max-width:1200px!important;margin:0 auto!important}
footer{opacity:.5}
/* hero */
.ma-hero{position:relative;overflow:hidden;border-radius:24px;padding:40px 40px 32px;color:#fff;
  background:radial-gradient(1200px 400px at 0% 0%,#4f46e5 0%,transparent 60%),
             radial-gradient(900px 400px at 100% 100%,#c026d3 0%,transparent 55%),#1e1b4b;
  box-shadow:0 20px 50px -20px rgba(79,70,229,.55)}
.ma-hero:after{content:"";position:absolute;inset:0;background-image:radial-gradient(rgba(255,255,255,.12) 1px,transparent 1px);
  background-size:22px 22px;mask-image:linear-gradient(180deg,rgba(0,0,0,.6),transparent);pointer-events:none}
.ma-kicker{display:inline-flex;gap:8px;align-items:center;font-size:12px;font-weight:600;letter-spacing:.08em;text-transform:uppercase;
  padding:6px 12px;border-radius:999px;background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.2)}
.ma-hero h1{font-size:clamp(28px,4vw,42px)!important;line-height:1.1;font-weight:800;margin:16px 0 10px!important;color:#fff!important;letter-spacing:-.02em}
.ma-hero h1 span{background:linear-gradient(90deg,#c7d2fe,#f5d0fe);-webkit-background-clip:text;background-clip:text;color:transparent}
.ma-hero p{font-size:16px;opacity:.85;max-width:680px;margin:0 0 24px!important;color:#e0e7ff!important}
.ma-flow{display:grid;grid-template-columns:1fr auto 1fr auto 1fr;gap:10px;align-items:center;position:relative;z-index:1}
.ma-flow-step{display:flex;gap:12px;align-items:center;padding:14px 16px;border-radius:14px;background:rgba(255,255,255,.08);
  border:1px solid rgba(255,255,255,.16);backdrop-filter:blur(6px)}
.ma-flow-n{flex:none;width:30px;height:30px;border-radius:10px;display:grid;place-items:center;font-weight:700;background:rgba(255,255,255,.18)}
.ma-hero .ma-flow-step,.ma-hero .ma-flow-step b,.ma-hero .ma-flow-step small,.ma-hero .ma-flow-arrow,.ma-hero .ma-kicker{color:#fff!important}
.ma-flow-step b{display:block;font-size:14px}.ma-flow-step small{font-size:12px;opacity:.75}
.ma-bare{background:none!important;border:none!important;box-shadow:none!important;padding:0!important}
.ma-flow-arrow{opacity:.6;font-size:18px}
@media (max-width:860px){.ma-hero{padding:28px 20px}.ma-flow{grid-template-columns:1fr}.ma-flow-arrow{display:none}}
/* section titles */
.ma-h{display:flex;align-items:center;gap:10px;font-weight:700;font-size:18px;margin:8px 2px 0}
.ma-h small{font-weight:500;font-size:13px;color:var(--body-text-color-subdued)}
.ma-card-col{background:var(--block-background-fill);border:1px solid var(--block-border-color);border-radius:20px;padding:18px!important;
  box-shadow:0 1px 2px rgba(16,24,40,.04),0 8px 24px rgba(16,24,40,.06)}
#ma-start{font-size:17px;font-weight:700;letter-spacing:.01em;box-shadow:0 10px 30px -10px rgba(124,58,237,.7);transition:transform .15s}
#ma-start:hover{transform:translateY(-1px)}
/* status */
.ma-status{display:flex;flex-direction:column;gap:12px}
.ma-progress{height:6px;border-radius:999px;background:var(--background-fill-secondary);overflow:hidden}
.ma-progress div{height:100%;border-radius:999px;background:linear-gradient(90deg,#6366f1,#a855f7,#d946ef);transition:width .6s ease}
.ma-bar-fail div{background:#ef4444}
.ma-steps{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}
@media (max-width:860px){.ma-steps{grid-template-columns:1fr}}
.ma-step{padding:16px;border-radius:16px;background:var(--block-background-fill);border:1px solid var(--block-border-color);transition:all .3s}
.ma-step-top{display:flex;justify-content:space-between;align-items:center;margin-bottom:12px}
.ma-dot{width:34px;height:34px;border-radius:12px;display:grid;place-items:center;font-weight:700;font-size:15px;
  background:var(--background-fill-secondary);color:var(--body-text-color-subdued)}
.ma-badge{font-size:12px;font-weight:600;padding:4px 10px;border-radius:999px;background:var(--background-fill-secondary);color:var(--body-text-color-subdued)}
.ma-step-label{font-weight:700;font-size:15px}
.ma-step-model{font-size:12.5px;color:var(--body-text-color-subdued);margin-top:2px;font-family:var(--font-mono)}
.ma-step-msg{margin-top:10px;font-size:13px;padding:8px 10px;border-radius:10px;background:var(--background-fill-secondary)}
.ma-running{border-color:#8b5cf6;box-shadow:0 0 0 4px rgba(139,92,246,.15)}
.ma-running .ma-dot{background:linear-gradient(135deg,#6366f1,#a855f7);color:#fff}
.ma-running .ma-badge{background:rgba(139,92,246,.15);color:#7c3aed}
.ma-completed .ma-dot{background:#10b981;color:#fff}.ma-completed .ma-badge{background:rgba(16,185,129,.14);color:#059669}
.ma-failed{border-color:#ef4444}.ma-failed .ma-dot{background:#ef4444;color:#fff}.ma-failed .ma-badge{background:rgba(239,68,68,.14);color:#dc2626}
.ma-skipped{opacity:.6}
.ma-spin{width:16px;height:16px;border:2.5px solid rgba(255,255,255,.4);border-top-color:#fff;border-radius:50%;animation:ma-spin .8s linear infinite}
@keyframes ma-spin{to{transform:rotate(360deg)}}
.ma-alert{padding:12px 16px;border-radius:14px;font-size:14px;border:1px solid transparent}
.ma-error{background:rgba(239,68,68,.08);border-color:rgba(239,68,68,.3);color:#dc2626}
.ma-warn{background:rgba(245,158,11,.08);border-color:rgba(245,158,11,.3);color:#b45309}
.ma-ok{background:rgba(16,185,129,.08);border-color:rgba(16,185,129,.3);color:#047857}
.dark .ma-error{color:#fca5a5}.dark .ma-warn{color:#fcd34d}.dark .ma-ok{color:#6ee7b7}
/* stats */
.ma-stats{display:grid;grid-template-columns:repeat(5,1fr);gap:12px}
@media (max-width:860px){.ma-stats{grid-template-columns:repeat(2,1fr)}}
.ma-stat{padding:16px 18px;border-radius:16px;background:var(--block-background-fill);border:1px solid var(--block-border-color)}
.ma-stat-v{font-size:26px;font-weight:800;letter-spacing:-.02em;background:linear-gradient(90deg,#6366f1,#a855f7);
  -webkit-background-clip:text;background-clip:text;color:transparent}
.ma-stat-k{font-size:12.5px;color:var(--body-text-color-subdued);font-weight:500;margin-top:2px}
/* tabs */
#ma-tabs>div:first-child button{font-weight:600;font-size:14.5px;padding:10px 16px}
#ma-tabs button.selected{color:#6366f1}
/* transcripts */
.ma-legend{display:flex;gap:16px;flex-wrap:wrap;font-size:13px;color:var(--body-text-color-subdued);margin:4px 2px 12px}
.ma-panes{display:grid;grid-template-columns:1fr 1fr;gap:14px}
@media (max-width:860px){.ma-panes{grid-template-columns:1fr}}
.ma-pane{border:1px solid var(--block-border-color);border-radius:16px;min-width:0;overflow:hidden;background:var(--block-background-fill)}
.ma-pane-h{display:flex;align-items:center;gap:10px;padding:12px 16px;font-weight:600;font-size:14px;border-bottom:1px solid var(--block-border-color);
  background:var(--background-fill-secondary)}
.ma-pill{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;padding:3px 9px;border-radius:999px}
.ma-pill-raw{background:rgba(100,116,139,.15);color:#64748b}.ma-pill-ref{background:rgba(99,102,241,.15);color:#6366f1}
.ma-tx{max-height:560px;overflow:auto;padding:8px 16px 14px;font-size:14px;line-height:1.65}
.ma-line{padding:10px 0;border-bottom:1px dashed var(--block-border-color)}
.ma-line:last-child{border-bottom:none}
.ma-meta{display:flex;align-items:center;gap:8px;margin-bottom:4px}
.ma-ts{font-family:var(--font-mono);font-size:11.5px;color:var(--body-text-color-subdued)}
.ma-spk{font-size:11.5px;font-weight:700;padding:2px 8px;border-radius:999px;color:var(--c);
  background:color-mix(in srgb,var(--c) 14%,transparent)}
.ma-text{word-break:break-word}
mark.ma-del{background:rgba(239,68,68,.16);color:inherit;border-radius:4px;padding:0 2px;text-decoration:line-through;text-decoration-color:rgba(239,68,68,.6)}
mark.ma-ins{background:rgba(16,185,129,.2);color:inherit;border-radius:4px;padding:0 2px}
.ma-empty-in{padding:24px 0;color:var(--body-text-color-subdued);font-style:italic}
/* cards, lists */
.ma-card{padding:24px 28px;border-radius:16px;background:var(--block-background-fill);border:1px solid var(--block-border-color)}
.ma-prose p{font-size:15.5px;line-height:1.75;margin:0 0 14px}.ma-prose p:last-child{margin:0}
.ma-prose ul{margin:0 0 14px 18px}.ma-prose li{margin:6px 0;line-height:1.65}
.ma-list{display:flex;flex-direction:column;gap:10px}
.ma-item{display:flex;gap:14px;align-items:flex-start;padding:16px 18px;border-radius:16px;background:var(--block-background-fill);
  border:1px solid var(--block-border-color);transition:border-color .2s,transform .2s}
.ma-item:hover{border-color:#a5b4fc;transform:translateY(-1px)}
.ma-num{flex:none;width:30px;height:30px;border-radius:10px;display:grid;place-items:center;font-weight:700;font-size:13px;color:#fff;
  background:linear-gradient(135deg,#6366f1,#a855f7)}
.ma-item-t{font-size:15px;line-height:1.6;font-weight:500;padding-top:3px}
.ma-item-body{display:flex;flex-direction:column;gap:10px}
.ma-chips{display:flex;gap:8px;flex-wrap:wrap}
.ma-chip{font-size:12.5px;font-weight:600;padding:4px 10px;border-radius:999px;background:rgba(99,102,241,.1);color:#4f46e5}
.dark .ma-chip{color:#a5b4fc}
.ma-chip-none{background:var(--background-fill-secondary);color:var(--body-text-color-subdued)}
.ma-empty{text-align:center;padding:48px 20px;border:1.5px dashed var(--block-border-color);border-radius:18px}
.ma-empty-icon{font-size:34px;margin-bottom:8px}.ma-empty-t{font-weight:700;font-size:16px}
.ma-empty-s{font-size:13.5px;color:var(--body-text-color-subdued);margin-top:4px}
.ma-dl{font-size:14px;color:var(--body-text-color-subdued);margin:2px 2px 10px}
"""

HERO = f"""
<div class="ma-hero">
  <span class="ma-kicker">🎙️ AI Meeting Assistant</span>
  <h1>Turn any recording into<br><span>minutes, decisions &amp; action items</span></h1>
  <p>Upload an English meeting recording. Three models run one after another, and every intermediate result
     stays visible and downloadable.</p>
  <div class="ma-flow">
    <div class="ma-flow-step"><div class="ma-flow-n">1</div><div><b>Speech-to-text</b><small>WhisperX {config.WHISPER_MODEL} · pyannote diarization</small></div></div>
    <div class="ma-flow-arrow">→</div>
    <div class="ma-flow-step"><div class="ma-flow-n">2</div><div><b>Refinement · LLM 1</b><small>{config.REFINE_MODEL} (Gemini)</small></div></div>
    <div class="ma-flow-arrow">→</div>
    <div class="ma-flow-step"><div class="ma-flow-n">3</div><div><b>Meeting record · LLM 2</b><small>{config.LLM2_MODEL} (OpenRouter)</small></div></div>
  </div>
</div>
"""

# Gradio 6 takes css/theme in launch(); older versions take them in Blocks().
_style = {"css": CSS, "theme": THEME}
_GR6 = int(gr.__version__.split(".")[0]) >= 6
LAUNCH_KW = _style if _GR6 else {}

with gr.Blocks(title="Meeting Assistant", **({} if _GR6 else _style)) as demo:
    gr.HTML(HERO, elem_classes="ma-bare")
    with gr.Row(equal_height=True):
        with gr.Column(scale=5, elem_classes="ma-card-col"):
            gr.HTML("<div class='ma-h'>1 · Upload <small>audio or video, up to 500 MB</small></div>")
            audio = gr.File(label="Meeting recording", type="filepath", file_types=sorted(ALLOWED_EXT), height=170)
        with gr.Column(scale=6, elem_classes="ma-card-col"):
            gr.HTML("<div class='ma-h'>2 · Help the models <small>optional</small></div>")
            context = gr.Textbox(label="Meeting context", lines=3, max_lines=6,
                                 placeholder="Participant names, product names, acronyms, jargon.\n"
                                             "E.g. Participants: Laura, David, Andrew, Craig. Topic: remote control design.")
            speakers = gr.Slider(0, 10, value=0, step=1, label="Number of speakers",
                                 info="Leave at 0 to detect automatically (2 to 8).")
    start_btn = gr.Button("✨  Start processing", variant="primary", size="lg", elem_id="ma-start")

    gr.HTML("<div class='ma-h'>Progress</div>")
    status = gr.HTML(status_html(blank_state()))
    overview = gr.HTML()

    gr.HTML("<div class='ma-h'>Results</div>")
    with gr.Tabs(elem_id="ma-tabs"):
        with gr.Tab("📝 Transcripts"):
            gr.HTML("<div class='ma-legend'><span><mark class='ma-del'>struck red</mark> raw words that were changed</span>"
                    "<span><mark class='ma-ins'>green</mark> corrected words in the refined transcript</span></div>")
            transcripts = gr.HTML(empty_html("🎙️", "No transcript yet",
                                             "The raw and refined transcripts appear here side by side."))
        with gr.Tab("📋 Minutes"):
            minutes_box = gr.HTML(minutes_html(None))
        with gr.Tab("⚖️ Key decisions"):
            decisions_box = gr.HTML(decisions_html(None))
        with gr.Tab("🎯 Action items"):
            actions_box = gr.HTML(actions_html(None))
        with gr.Tab("⬇️ Downloads"):
            gr.HTML("<div class='ma-dl'>A zip with everything, plus each file: raw transcript (.txt, .json), refined "
                    "transcript (.txt), and the meeting record as human-readable .md / .txt and machine-readable .json.</div>")
            downloads = gr.File(label="Output files", file_count="multiple", interactive=False)

    start_btn.click(process, inputs=[audio, context, speakers],
                    outputs=[status, overview, transcripts, minutes_box, decisions_box, actions_box, downloads],
                    concurrency_limit=1, show_progress="hidden")

print("Interface built. Run the next cell to launch it.")

/tmp/ipykernel_15741/2895351007.py:496: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Meeting Assistant", **({} if _GR6 else _style)) as demo:
/tmp/ipykernel_15741/2895351007.py:496: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Meeting Assistant", **({} if _GR6 else _style)) as demo:


Interface built. Run the next cell to launch it.


In [7]:
# ============================================================
# STEP 7 — LAUNCH THE APP
# ============================================================
# Opens the app below and prints a public https://….gradio.live link (valid while this cell runs).
# Stop it with the ■ button on this cell.
demo.queue().launch(share=True, debug=True, allowed_paths=[str(OUT_ROOT)], **LAUNCH_KW)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e62d45143dc047038f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


2026-10-07 21:56:45 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

2026-10-07 21:57:33 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e62d45143dc047038f.gradio.live
